# 04 — Synthetic ground-truth skeleton recovery

This benchmark validates the production 3-D skeleton-to-graph path against the historical `poly2graph` backend on the same synthetic spatial graphs. Every admissible volume is voxelized and Lee-skeletonized **once**, then the identical skeleton is supplied to both extractors.

The reconstruction suite uses connected, bridgeless, exactly trivalent ground-truth graphs and tests abstract graph recovery after the normal cleanup stage. The optimized extractor receives `max_junction_degree=3`, which enables its fail-closed multi-scale junction repair.

The Yamada invariant is **not** used to decide trivalent reconstruction success. A separate final check evaluates Yamada only on maximum-degree-2 embedded cycles, so the invariant check is cleanly separated from graph-reconstruction accuracy.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
import statistics, sys, time
import networkx as nx
import numpy as np
import sympy as sp
from skimage.morphology import ball, dilation, skeletonize

ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'pyproject.toml').exists(): ROOT=ROOT.parent
if not (ROOT/'pyproject.toml').exists(): raise RuntimeError('Run inside the KnottedGraph checkout.')

import knotted_graph
from knotted_graph.core import contract_short_edges, remove_leaf_nodes, simplify_edges, smooth_edges
from knotted_graph.extraction import skeleton_image_to_graph
from knotted_graph.invariants.yamada.native import native_available, native_import_error
from knotted_graph.projection import compute_yamada_polynomial

A=sp.Symbol('A')
BOUND=1.35
N=200
RADII=[1,2,3]
TRANSFORMS=['identity','rotate','affine']
CLEARANCE_FRACTION=.40
DX=2*BOUND/(N-1)
print('Python:',sys.executable)
print('KnottedGraph:',Path(knotted_graph.__file__).resolve())
print('Native Yamada backend:',native_available())
print('Native import error:',native_import_error())
print(f'N={N}, dx={DX:.6f}')

In [ ]:
@dataclass
class Case:
    name:str
    graph:nx.MultiGraph
    radius_cap:float

def normalize(X,scale=.72):
    X=np.asarray(X,float); X-=X.mean(0)
    return X*(scale/np.max(np.linalg.norm(X,axis=1)))

def embedded_graph(pos,edges):
    H=nx.MultiGraph()
    for n,p in pos.items(): H.add_node(n,pos=np.asarray(p,float))
    for u,v,P in edges: H.add_edge(u,v,pts=np.asarray(P,float))
    return H

def theta_case(name,bowed=False,n=500):
    t=np.linspace(0,1,n); x=-.72+1.44*t
    if bowed:
        C=[np.c_[x,-.58*np.sin(np.pi*t), .16*np.sin(2*np.pi*t)],
           np.c_[x, .10*np.sin(2*np.pi*t),-.10*np.sin(np.pi*t)],
           np.c_[x, .58*np.sin(np.pi*t),-.16*np.sin(2*np.pi*t)]]
    else:
        C=[np.c_[x,-.58*np.sin(np.pi*t),0*t],np.c_[x,0*t,0*t],np.c_[x,.58*np.sin(np.pi*t),0*t]]
    for P in C: P[0]=[-.72,0,0]; P[-1]=[.72,0,0]
    return Case(name,embedded_graph({'u':C[0][0],'v':C[0][-1]},[('u','v',P) for P in C]),.060)

def segdist(p1,q1,p2,q2):
    u=q1-p1; v=q2-p2; w=p1-p2
    a=u@u; b=u@v; c=v@v; d=u@w; e=v@w; D=a*c-b*b
    if D<1e-14: s=0.; t=np.clip(e/c if c>1e-14 else 0.,0,1)
    else: s=np.clip((b*e-c*d)/D,0,1); t=np.clip((a*e-b*d)/D,0,1)
    if a>1e-14: s=np.clip((b*t-d)/a,0,1)
    if c>1e-14: t=np.clip((b*s+e)/c,0,1)
    return float(np.linalg.norm(w+s*u-t*v))

def straight_clearance(G,P):
    E=list(G.edges()); best=np.inf
    for i,(u,v) in enumerate(E):
        for a,b in E[i+1:]:
            if {u,v}&{a,b}: continue
            best=min(best,segdist(P[u],P[v],P[a],P[b]))
    return best

def cubic_case(name,G,planar,seed,cap):
    G=nx.Graph(G)
    assert nx.is_connected(G) and not list(nx.bridges(G)) and all(d==3 for _,d in G.degree())
    if planar:
        ok,_=nx.check_planarity(G); assert ok
        p=nx.planar_layout(G); X=normalize([[p[n][0],p[n][1],0.] for n in G]); P={n:X[i] for i,n in enumerate(G)}
    else:
        P=None
        for trial in range(250):
            p=nx.spring_layout(G,dim=3,seed=seed+trial,iterations=700)
            X=normalize([p[n] for n in G]); Q={n:X[i] for i,n in enumerate(G)}
            if straight_clearance(G,Q)>.055: P=Q; break
        if P is None: raise RuntimeError(f'Could not find clear embedding for {name}')
    return Case(name,embedded_graph(P,[(u,v,np.linspace(P[u],P[v],100)) for u,v in G.edges()]),cap)

CASES=[
 theta_case('theta3_planar'),theta_case('theta3_bowed',True),
 cubic_case('K4',nx.complete_graph(4),True,11,.052),
 cubic_case('triangular_prism',nx.circular_ladder_graph(3),True,12,.045),
 cubic_case('cube',nx.cubical_graph(),True,13,.042),
 cubic_case('pentagonal_prism',nx.circular_ladder_graph(5),True,14,.035),
 cubic_case('dodecahedral',nx.dodecahedral_graph(),True,15,.027),
 cubic_case('K3_3',nx.complete_bipartite_graph(3,3),False,17,.032),
 cubic_case('petersen',nx.petersen_graph(),False,18,.030),
 cubic_case('heawood',nx.heawood_graph(),False,19,.024),
]
for c in CASES:
    assert set(dict(c.graph.degree()).values())=={3}
    print(c.name,c.graph.number_of_nodes(),c.graph.number_of_edges())

In [ ]:
def Rxyz(a,b,c):
    a,b,c=np.deg2rad([a,b,c])
    Rx=np.array([[1,0,0],[0,np.cos(a),-np.sin(a)],[0,np.sin(a),np.cos(a)]])
    Ry=np.array([[np.cos(b),0,np.sin(b)],[0,1,0],[-np.sin(b),0,np.cos(b)]])
    Rz=np.array([[np.cos(c),-np.sin(c),0],[np.sin(c),np.cos(c),0],[0,0,1]])
    return Rz@Ry@Rx

def transform(name):
    if name=='identity': M,b=np.eye(3),np.zeros(3)
    elif name=='rotate': M,b=Rxyz(21,34,13),np.array([.04,-.03,.02])
    elif name=='affine':
        M=Rxyz(17,-23,31)@np.diag([1.08,.91,1.03])@np.array([[1,.13,0],[0,1,.09],[.05,0,1]])
        b=np.array([-.03,.04,-.02])
    else: raise ValueError(name)
    assert np.linalg.det(M)>0
    return M,b

def deform(G,name):
    M,b=transform(name); H=nx.MultiGraph()
    for n,d in G.nodes(data=True): H.add_node(n,pos=np.asarray(d['pos'])@M.T+b)
    for u,v,k,d in G.edges(keys=True,data=True): H.add_edge(u,v,pts=np.asarray(d['pts'])@M.T+b)
    return H

def trimmed(P,f=.15):
    P=np.asarray(P,float); n=max(1,int(round(f*len(P))))
    return P[n:-n] if 2*n<len(P) else P

def interior_sep(G):
    E=[(u,v,np.asarray(d['pts'],float)) for u,v,k,d in G.edges(keys=True,data=True)]; best=np.inf
    for i,(u,v,P0) in enumerate(E):
        for a,b,Q0 in E[i+1:]:
            P,Q=(trimmed(P0),trimmed(Q0)) if {u,v}&{a,b} else (P0,Q0)
            for s in range(0,len(P),128):
                D2=np.sum((P[s:s+128,None,:]-Q[None,:,:])**2,axis=-1); best=min(best,float(np.sqrt(D2.min())))
    return best

def admissible(case,G,r):
    rw=r*DX; sep=interior_sep(G); limit=min(case.radius_cap,CLEARANCE_FRACTION*sep)
    return rw<=limit

def resample(P,step):
    P=np.asarray(P,float); parts=[]
    for p,q in zip(P[:-1],P[1:]):
        n=max(2,int(np.ceil(np.linalg.norm(q-p)/step))+1); parts.append(np.linspace(p,q,n,endpoint=False))
    parts.append(P[-1:]); return np.vstack(parts)

def voxelize(G,r):
    V=np.zeros((N,N,N),bool)
    for _,_,_,d in G.edges(keys=True,data=True):
        P=resample(d['pts'],DX/3); I=np.rint((P+BOUND)/(2*BOUND)*(N-1)).astype(int); I=np.clip(I,0,N-1)
        V[I[:,0],I[:,1],I[:,2]]=1
    return dilation(V,footprint=ball(r))

def to_world(H):
    H=nx.MultiGraph(H); origin=np.array([-BOUND]*3,float)
    for _,d in H.nodes(data=True): d['pos']=origin+DX*np.asarray(d['pos'],float)
    for _,_,_,d in H.edges(keys=True,data=True): d['pts']=origin+DX*np.asarray(d['pts'],float)
    return H

def cleanup_baseline(H):
    H=remove_leaf_nodes(H); H=simplify_edges(H)
    H=contract_short_edges(H,min_length=2.5*DX,copy=False)
    H=remove_leaf_nodes(H); H=simplify_edges(H)
    return smooth_edges(H,epsilon=2*DX,copy=False)

def cleanup_optimized(H):
    H=remove_leaf_nodes(H); H=simplify_edges(H)
    return smooth_edges(H,epsilon=2*DX,copy=False)

In [ ]:
records=[]
for case in CASES:
    for transform_name in TRANSFORMS:
        target=deform(case.graph,transform_name)
        for radius in RADII:
            if not admissible(case,target,radius):
                continue
            volume=voxelize(target,radius)
            t0=time.perf_counter(); skeleton=skeletonize(volume,method='lee'); skeleton_time=time.perf_counter()-t0

            baseline_times=[]; optimized_times=[]
            for _ in range(3):
                t0=time.perf_counter(); B=skeleton_image_to_graph(skeleton,backend='poly2graph'); baseline_times.append(time.perf_counter()-t0)
                t0=time.perf_counter(); O=skeleton_image_to_graph(skeleton,backend='topology_aware',max_junction_degree=3); optimized_times.append(time.perf_counter()-t0)

            Bc=cleanup_baseline(to_world(B)); Oc=cleanup_optimized(to_world(O))
            b_ok=nx.is_isomorphic(nx.MultiGraph(case.graph),Bc)
            o_ok=nx.is_isomorphic(nx.MultiGraph(case.graph),Oc)
            row=dict(case=case.name,transform=transform_name,radius=radius,baseline=b_ok,optimized=o_ok,
                     baseline_extract=statistics.median(baseline_times),optimized_extract=statistics.median(optimized_times),skeleton_time=skeleton_time)
            records.append(row)
            print(row)

n=len(records); bp=sum(r['baseline'] for r in records); op=sum(r['optimized'] for r in records)
bt=statistics.median(r['baseline_extract'] for r in records); ot=statistics.median(r['optimized_extract'] for r in records)
print(f'ADMISSIBLE={n} BASELINE={bp}/{n} OPTIMIZED={op}/{n}')
print(f'MEDIAN EXTRACT baseline={1e3*bt:.3f} ms optimized={1e3*ot:.3f} ms speedup={bt/ot:.3f}x')
assert n>=30
assert op==n
assert op-bp>=10
assert ot<bt
assert bt/ot>=1.25

## Degree-$\le 2$ Yamada deformation check

This check is deliberately separate from reconstruction. It uses only embedded cycles (maximum graph degree 2) and verifies invariance under an orientation-preserving affine deformation.

In [ ]:
def cycle_graph(offset=(0,0,0),radius=.6,n=180):
    t=np.linspace(0,2*np.pi,n,endpoint=True)
    P=np.c_[radius*np.cos(t),radius*np.sin(t),np.zeros_like(t)]+np.asarray(offset,float)
    P[-1]=P[0]
    G=nx.MultiGraph(); G.add_node(0,pos=P[0].copy()); G.add_edge(0,0,pts=P)
    return G

def affine_embedded(G):
    M=np.array([[1.05,.12,.03],[.02,.94,.08],[.04,.01,1.02]])@Rxyz(13,21,8); b=np.array([.08,-.05,.04])
    assert np.linalg.det(M)>0
    H=nx.MultiGraph()
    for n,d in G.nodes(data=True): H.add_node(n,pos=np.asarray(d['pos'])@M.T+b)
    for u,v,k,d in G.edges(keys=True,data=True): H.add_edge(u,v,pts=np.asarray(d['pts'])@M.T+b)
    return H

for label,G in [('unknot_cycle',cycle_graph()),('two_cycles',nx.disjoint_union(cycle_graph(offset=(-.8,0,0),radius=.35),cycle_graph(offset=(.8,0,0),radius=.35)))]:
    assert max(dict(G.degree()).values())<=2
    H=affine_embedded(G)
    assert max(dict(H.degree()).values())<=2
    y0=sp.expand(compute_yamada_polynomial(G,A,n_jobs=1))
    y1=sp.expand(compute_yamada_polynomial(H,A,n_jobs=1))
    print(label,y0,y1)
    assert sp.expand(y0-y1)==0
print('Yamada degree<=2 deformation checks: PASS')

## Acceptance result

The notebook passes only if the optimized extractor recovers every admissible synthetic graph, improves by at least ten cases over the historical pipeline, is at least 1.25× faster in median extraction time, and the separate maximum-degree-2 Yamada deformation checks agree exactly.